# Phase 02 — Label Schema and Baseline Definitions

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Define transparent labels and baseline models before building stronger training experiments.

This notebook is the Phase 2 source of truth. It writes `reports/phase_02_label_schema_baselines.json` so pair generation, normalization, baseline evaluation, and later training use the same label components, score bands, and validation controls.

## Purpose
Document and verify Phase 02 — Label Schema and Baseline Definitions in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 02.label.schema.baselines notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 02 — Label Schema and Baseline Definitions.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Contract boundary

Model/training owns transparent core signals only:

- `jobFitAlignment.score` as a `0-100` score derived from documented label components.
- `jobFitAlignment` evidence signals such as matched skills, missing skills, role match, experience match, requirement coverage, and confidence notes.
- `atsFriendliness.score` as a `0-100` score derived from parseability, structure, contact/date evidence, metric evidence, and formatting risk.
- `atsFriendliness.detectedIssues` as grounded issue identifiers from the ATS taxonomy.
- `overallImpression` inputs as grounded signal summaries, not invented product copy.

Backend/API wrapper still owns auth, file ownership, persistence, final response shaping, `topActionables`, `sectionReviews`, job detail hydration, and OpenAI wrapper orchestration. Optional location or work-preference signals are not part of the core job-fit label unless the backend supplies explicit candidate/preference context and a later ranking phase approves the contract.


## Shared setup

### Purpose
Define deterministic paths, reusable report helpers, score constraints, and validation checks used by every Phase 2 step.

### Required input
Repository root with `TODOS.md`, `reports/phase_01_data_audit_contracts.json`, `references/docs/generated/openapi.json`, and the legacy data snapshot documented by earlier phases.

### Action
Load only standard-library helpers, read prior contract evidence, and define static policy tables. No model training, pair generation, label mutation, or artifact overwrite occurs in this notebook.

### Expected output
Shared constants and validation helpers for label components, score bands, baseline definitions, manual validation sampling, and final report writing.

### Verification
Setup must run without optional notebook dependencies. Generated output must be limited to `reports/phase_02_label_schema_baselines.json`.


In [37]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "training").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd().resolve())
REPORTS = ROOT / "reports"
PHASE1_REPORT = REPORTS / "phase_01_data_audit_contracts.json"
PHASE2_REPORT = REPORTS / "phase_02_label_schema_baselines.json"
OPENAPI_JSON = ROOT / "references" / "docs" / "generated" / "openapi.json"

SCORE_RANGE = {
    "api_min": 0,
    "api_max": 100,
    "normalized_min": 0.0,
    "normalized_max": 1.0,
    "api_contract": "integer 0-100",
    "training_contract": "float 0.0-1.0 before final API scaling",
}


def load_json(path: Path) -> dict[str, Any]:
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def print_table(rows: list[dict[str, Any]], columns: list[str], max_rows: int = 20) -> None:
    shown = rows[:max_rows]
    if not shown:
        print("<empty>")
        return
    widths = {column: len(column) for column in columns}
    for row in shown:
        for column in columns:
            widths[column] = max(widths[column], len(str(row.get(column, ""))))
    print(" | ".join(column.ljust(widths[column]) for column in columns))
    print("-+-".join("-" * widths[column] for column in columns))
    for row in shown:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))
    if len(rows) > max_rows:
        print(f"... {len(rows) - max_rows} more rows")


def assert_weight_sum(rows: list[dict[str, Any]], expected: float = 1.0) -> float:
    total = round(sum(float(row["weight"]) for row in rows), 6)
    assert total == expected, f"Expected weight sum {expected}, got {total}"
    return total


phase1_report = load_json(PHASE1_REPORT)
openapi = load_json(OPENAPI_JSON)
openapi_score_constraints = phase1_report.get("openapi_score_constraints", {})

assert openapi_score_constraints["jobFitAlignment.score"]["minimum"] == 0
assert openapi_score_constraints["jobFitAlignment.score"]["maximum"] == 100
assert openapi_score_constraints["atsFriendliness.score"]["minimum"] == 0
assert openapi_score_constraints["atsFriendliness.score"]["maximum"] == 100

print(f"Repository root: {ROOT}")
print(f"Phase 1 readiness: {phase1_report['data_readiness_decision']['decision']}")
print("OpenAPI score constraints:")
print(json.dumps(openapi_score_constraints, indent=2))


Repository root: /Users/macbookpro/Development/bisakerja-model
Phase 1 readiness: Proceed to Phase 2 label schema and baseline definitions; do not start new model training yet.
OpenAPI score constraints:
{
  "jobFitAlignment.score": {
    "type": "integer",
    "minimum": 0,
    "maximum": 100
  },
  "atsFriendliness.score": {
    "type": "integer",
    "minimum": 0,
    "maximum": 100
  },
  "jobRecommendations[].matchScore": {
    "type": "integer",
    "minimum": 0,
    "maximum": 100
  }
}


## Step 2.1 — Job-fit label decomposition

### Purpose
Define skill overlap, semantic similarity, experience match, role match, requirement coverage, and optional location or work-preference components.

### Required input
Use audited Phase 1 field decisions, legacy pair fields, source job/profile text, normalized skills, normalized experience, role/title text, and requirement text. Optional preference data may only be used when the backend provides explicit, non-leaky user preference context.

### Action
Define a label schema draft with component name, weight, source evidence, scoring rule, missing-data policy, output evidence, and leakage control. Keep the core job-fit score independent from wrapper-only fields and identifiers.

### Expected output
A reviewable job-fit component table whose core component weights sum to `1.0`, plus a documented optional preference component that is disabled for the core scorer until a later candidate-ranking contract allows it.

### Verification
Every required component must be present before pair generation. Core components must sum to `1.0`. No identifier, generated label, hydrated job detail, application outcome, or wrapper copy may be used as an input feature.


In [38]:
jobfit_label_components = [
    {
        "component": "skill_overlap",
        "weight": 0.30,
        "direction": "higher_is_better",
        "source_evidence": "normalized profile/CV skills and normalized job requirements skills",
        "scoring_rule": "Jaccard-style exact overlap after Phase 3 skill normalization; score 0 when either side has no reliable skill evidence.",
        "missing_policy": "Record empty_skills flag and use 0.0 until skill extraction is fixed.",
        "model_evidence_output": "matchedSkills, missingSkills",
        "leakage_control": "Do not use fit_score, profile_id, job_id, company name, bookmarked/applied status, or validation labels.",
    },
    {
        "component": "semantic_similarity",
        "weight": 0.20,
        "direction": "higher_is_better",
        "source_evidence": "profile/CV text embedding and job text embedding",
        "scoring_rule": "Cosine similarity scaled to 0.0-1.0 after text construction policy is versioned.",
        "missing_policy": "Record empty_text flag and use 0.0 if either text side is empty after parsing.",
        "model_evidence_output": "summarySignals, confidenceNotes",
        "leakage_control": "Build text from source profile/CV/job fields only, not generated summaries or labels.",
    },
    {
        "component": "experience_match",
        "weight": 0.15,
        "direction": "higher_is_better",
        "source_evidence": "normalized candidate experience and normalized job seniority/experience requirement",
        "scoring_rule": "Score 1.0 for exact or above-minimum match, degrade by bounded gap size, score 0.0 for clear underqualification beyond allowed tolerance.",
        "missing_policy": "Record unknown_experience flag and use neutral 0.5 only for label drafting; Phase 3 must replace silent fallback with explicit unknown handling.",
        "model_evidence_output": "experienceMatch, confidenceNotes",
        "leakage_control": "Do not infer seniority from target label or post-hoc model prediction.",
    },
    {
        "component": "role_match",
        "weight": 0.15,
        "direction": "higher_is_better",
        "source_evidence": "normalized job title/category and candidate target role/profile title/CV headline when available",
        "scoring_rule": "Exact role-family match receives high score, adjacent role-family receives medium score, unrelated role receives low score.",
        "missing_policy": "Record unknown_role flag and use neutral 0.5 only when no role evidence exists.",
        "model_evidence_output": "roleMatch, summarySignals",
        "leakage_control": "Role family taxonomy must be derived before split labels are reviewed.",
    },
    {
        "component": "requirement_coverage",
        "weight": 0.20,
        "direction": "higher_is_better",
        "source_evidence": "job requirement summary, required skills, CV/profile evidence, and extracted project/experience claims",
        "scoring_rule": "Covered required evidence divided by required evidence count, with critical requirements allowed to carry higher local weight.",
        "missing_policy": "Record empty_requirements flag and exclude missing job requirements from denominator instead of rewarding unknown data.",
        "model_evidence_output": "matchedRequirements, missingRequirements, missingSignals",
        "leakage_control": "Requirement coverage cannot use wrapper-generated topActionables or sectionReviews.",
    },
]

optional_jobfit_components = [
    {
        "component": "preference_match_optional",
        "default_core_weight": 0.0,
        "allowed_future_weight_cap": 0.05,
        "source_evidence": "backend-provided user preferences such as location, work type, employment type, salary range, and candidate set constraints",
        "scoring_rule": "Use only as candidate-reranking metadata or bounded preference score after backend contract approval.",
        "missing_policy": "Do not penalize when preferences are absent; keep outside core jobFitAlignment label.",
        "owner": "backend_candidate_context_or_phase_09_reranking",
        "leakage_control": "Never hydrate job details from static model artifacts; backend owns visibility, freshness, and final job detail.",
    }
]

jobfit_weight_sum = assert_weight_sum(jobfit_label_components)
required_jobfit = {
    "skill_overlap",
    "semantic_similarity",
    "experience_match",
    "role_match",
    "requirement_coverage",
}
assert required_jobfit == {row["component"] for row in jobfit_label_components}

print(f"Core job-fit weight sum: {jobfit_weight_sum}")
print_table(
    jobfit_label_components,
    ["component", "weight", "source_evidence", "scoring_rule", "missing_policy"],
)
print("\nOptional component policy:")
print_table(optional_jobfit_components, ["component", "default_core_weight", "allowed_future_weight_cap", "owner"])


Core job-fit weight sum: 1.0
component            | weight | source_evidence                                                                                        | scoring_rule                                                                                                                              | missing_policy                                                                                                                                  
---------------------+--------+--------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------
skill_overlap        | 0.3    | normalized profile/CV skills and normalized job requirements skills                            

## Step 2.2 — ATS label decomposition

### Purpose
Define parseability, section completeness, contact detection, date detection, metric evidence, and formatting-risk components.

### Required input
Parsed CV text, file metadata, section detection output, contact/date regex evidence, quantified achievement evidence, and controlled CV benchmark cases. Current data does not yet contain a complete ATS benchmark, so this step defines the label schema and blocks production claims until Phase 7 evaluation exists.

### Action
Define an ATS label schema with component name, weight, evidence source, scoring rule, detected issue keys, missing-data policy, and verification requirement.

### Expected output
A reviewable ATS component table whose weights sum to `1.0` and whose issue keys can become `atsFriendliness.detectedIssues` without wrapper invention.

### Verification
Every ATS component must be explicitly measurable. Missing or empty parse results must lower confidence and create detected issues instead of producing unsupported positive scores.


In [39]:
ats_label_components = [
    {
        "component": "parseability",
        "weight": 0.30,
        "direction": "higher_is_better",
        "source_evidence": "text extraction result, extracted character count, page count, parser warnings",
        "scoring_rule": "Full readable text receives high score; partial parse receives medium score; empty/scanned/unreadable parse receives low score.",
        "detected_issue_keys": ["empty_text", "low_text_coverage", "scanned_or_image_only"],
        "missing_policy": "Empty parse is a hard issue and cannot receive high ATS score.",
    },
    {
        "component": "section_completeness",
        "weight": 0.20,
        "direction": "higher_is_better",
        "source_evidence": "detected sections such as summary, experience, education, skills, projects, certifications",
        "scoring_rule": "Score by required section coverage for target role; missing critical sections reduce score.",
        "detected_issue_keys": ["missing_skills_section", "missing_experience_section", "missing_education_section"],
        "missing_policy": "Unknown section headings require conservative score and review flag.",
    },
    {
        "component": "contact_detection",
        "weight": 0.15,
        "direction": "higher_is_better",
        "source_evidence": "email, phone, portfolio, LinkedIn/GitHub URL evidence where available",
        "scoring_rule": "Score by presence of reachable contact fields without exposing raw PII in labels.",
        "detected_issue_keys": ["missing_email", "missing_phone", "missing_portfolio_or_profile_link"],
        "missing_policy": "Do not store raw contact values in labels; store boolean evidence only.",
    },
    {
        "component": "date_detection",
        "weight": 0.10,
        "direction": "higher_is_better",
        "source_evidence": "employment/education date ranges and duration parse evidence",
        "scoring_rule": "Score by parseable date coverage for experience entries and education entries.",
        "detected_issue_keys": ["missing_dates", "ambiguous_dates", "inconsistent_date_order"],
        "missing_policy": "Unknown date formats are issues until locale-aware parsing is defined.",
    },
    {
        "component": "metric_evidence",
        "weight": 0.15,
        "direction": "higher_is_better",
        "source_evidence": "numbers, percentages, scale indicators, impact metrics, project outcomes in CV text",
        "scoring_rule": "Score by presence and density of quantified achievements in experience/project sections.",
        "detected_issue_keys": ["low_quantified_impact", "generic_responsibility_only"],
        "missing_policy": "Do not invent metrics; absence is a quality issue, not a reason to generate fake evidence.",
    },
    {
        "component": "formatting_risk",
        "weight": 0.10,
        "direction": "lower_risk_is_better",
        "source_evidence": "parser layout signals, table density, multi-column risk, excessive graphics, file type, text order anomalies",
        "scoring_rule": "Start from 1.0 and subtract bounded risk penalties for format patterns known to reduce ATS extraction reliability.",
        "detected_issue_keys": ["table_heavy_layout", "multi_column_risk", "excessive_graphics", "unsupported_file_type"],
        "missing_policy": "When layout evidence is unavailable, record unknown_format_risk and keep confidence low.",
    },
]

ats_weight_sum = assert_weight_sum(ats_label_components)
required_ats = {
    "parseability",
    "section_completeness",
    "contact_detection",
    "date_detection",
    "metric_evidence",
    "formatting_risk",
}
assert required_ats == {row["component"] for row in ats_label_components}

print(f"ATS weight sum: {ats_weight_sum}")
print_table(
    ats_label_components,
    ["component", "weight", "source_evidence", "scoring_rule", "detected_issue_keys"],
)


ATS weight sum: 1.0
component            | weight | source_evidence                                                                                              | scoring_rule                                                                                                                    | detected_issue_keys                                                                       
---------------------+--------+--------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------
parseability         | 0.3    | text extraction result, extracted character count, page count, parser warnings                               | Full readable text receives high score; partial parse receives medium score; empty/scanned/unreadable parse r

## Step 2.3 — Score band policy

### Purpose
Document low, medium, and high bands with numeric thresholds and explain what each band means for product users.

### Required input
OpenAPI `0-100` score constraints, Phase 0 weak-label distribution, Phase 1 output contract mapping, and product boundary requirements.

### Action
Define one shared draft band policy for normalized training scores and API scores. Use the same thresholds for job-fit, ATS, and candidate match until Phase 10 calibration proves better thresholds.

### Expected output
Numeric low, medium, and high bands with product-user meaning, model-training meaning, and release caveats.

### Verification
Bands must be contiguous, non-overlapping, cover `0-100`, map to `0.0-1.0`, and avoid production calibration claims.


In [40]:
score_band_policy = [
    {
        "band": "low",
        "normalized_min": 0.00,
        "normalized_max": 0.34,
        "api_min": 0,
        "api_max": 34,
        "match_level": "low",
        "product_meaning_jobfit": "Major skill, role, experience, or requirement gaps. User should treat the role as aspirational or needing substantial preparation.",
        "product_meaning_ats": "CV has serious parseability, structure, contact/date, metric, or formatting issues that can block reliable screening.",
        "training_requirement": "Pair generation must include true low cases and hard negatives, not only random weak pairs.",
    },
    {
        "band": "medium",
        "normalized_min": 0.35,
        "normalized_max": 0.64,
        "api_min": 35,
        "api_max": 64,
        "match_level": "medium",
        "product_meaning_jobfit": "Partial match with clear strengths and visible gaps. User may apply after targeted improvements.",
        "product_meaning_ats": "CV is mostly parseable but has issues that can reduce ranking or hide important evidence.",
        "training_requirement": "Pair generation must include adjacent-role and partial-skill cases, not only extremes.",
    },
    {
        "band": "high",
        "normalized_min": 0.65,
        "normalized_max": 1.00,
        "api_min": 65,
        "api_max": 100,
        "match_level": "high",
        "product_meaning_jobfit": "Strong match with most critical evidence present. Remaining gaps should be specific and actionable.",
        "product_meaning_ats": "CV is parseable, structured, contact/date evidence is detectable, and formatting risk is low.",
        "training_requirement": "Validation and training splits must contain high-fit/high-quality examples before any model can claim full-range scoring.",
    },
]

assert score_band_policy[0]["api_min"] == SCORE_RANGE["api_min"]
assert score_band_policy[-1]["api_max"] == SCORE_RANGE["api_max"]
for left, right in zip(score_band_policy, score_band_policy[1:]):
    assert left["api_max"] + 1 == right["api_min"]
    assert round(left["normalized_max"] + 0.01, 2) == right["normalized_min"]

score_policy_notes = {
    "calibration_status": "draft_policy_not_production_calibrated",
    "phase_10_requirement": "Replace or confirm these thresholds only after calibration diagnostics and score-bucket error are reported.",
    "current_snapshot_warning": "Legacy weak labels had no high-fit samples above the high-band threshold, so Phase 4 must create balanced pairs before training.",
}

print_table(
    score_band_policy,
    ["band", "normalized_min", "normalized_max", "api_min", "api_max", "match_level", "training_requirement"],
)
print("\nPolicy notes:")
print(json.dumps(score_policy_notes, indent=2))


band   | normalized_min | normalized_max | api_min | api_max | match_level | training_requirement                                                                                                     
-------+----------------+----------------+---------+---------+-------------+--------------------------------------------------------------------------------------------------------------------------
low    | 0.0            | 0.34           | 0       | 34      | low         | Pair generation must include true low cases and hard negatives, not only random weak pairs.                              
medium | 0.35           | 0.64           | 35      | 64      | medium      | Pair generation must include adjacent-role and partial-skill cases, not only extremes.                                   
high   | 0.65           | 1.0            | 65      | 100     | high        | Validation and training splits must contain high-fit/high-quality examples before any model can claim full-range scoring.

Poli

## Step 2.4 — Baseline model catalog

### Purpose
Define constant mean, constant median, skill-overlap-only, cosine-only, and simple regression baselines for future comparison.

### Required input
Train/validation split from Phase 4, label schema from this notebook, normalized features from Phase 3, embeddings/text features, and no validation labels during fitting.

### Action
Define baseline name, prediction target, allowed inputs, fitting rule, metrics, leakage controls, and expected use in Phase 5 evaluation.

### Expected output
A baseline catalog that Phase 5 can execute before any neural or feature-augmented model is accepted.

### Verification
Every complex model must be compared to the best valid baseline. Constant baselines must use train-only statistics. Feature baselines must not use validation labels, identifiers, wrapper-owned fields, or final product copy.


In [41]:
baseline_model_catalog = [
    {
        "baseline": "constant_mean",
        "target": "job_fit_label_v2 and ats_label_v1 when labels exist",
        "allowed_inputs": "training labels only",
        "fit_rule": "Predict the training-set mean for every validation example.",
        "required_metrics": ["MAE", "RMSE", "R2", "score_band_agreement"],
        "leakage_control": "Mean must be computed from training split only.",
        "acceptance_use": "Minimum sanity baseline; complex model must beat it materially.",
    },
    {
        "baseline": "constant_median",
        "target": "job_fit_label_v2 and ats_label_v1 when labels exist",
        "allowed_inputs": "training labels only",
        "fit_rule": "Predict the training-set median for every validation example.",
        "required_metrics": ["MAE", "RMSE", "R2", "score_band_agreement"],
        "leakage_control": "Median must be computed from training split only.",
        "acceptance_use": "Robust constant baseline for skewed labels.",
    },
    {
        "baseline": "skill_overlap_only",
        "target": "job_fit_label_v2",
        "allowed_inputs": "normalized candidate skills and normalized job required skills",
        "fit_rule": "Predict normalized skill-overlap score directly, optionally calibrated on training split only.",
        "required_metrics": ["MAE", "RMSE", "R2", "Spearman", "score_band_agreement", "slice_metrics"],
        "leakage_control": "No generated fit_score, profile_id, job_id, company, application outcome, or validation labels.",
        "acceptance_use": "Proves whether exact skill matching already explains labels.",
    },
    {
        "baseline": "cosine_only",
        "target": "job_fit_label_v2",
        "allowed_inputs": "profile/CV text embedding and job text embedding from versioned text construction",
        "fit_rule": "Predict clipped cosine similarity or train-only calibrated cosine score.",
        "required_metrics": ["MAE", "RMSE", "R2", "Spearman", "score_band_agreement", "slice_metrics"],
        "leakage_control": "Embedding text must exclude labels, wrapper summaries, and hydrated recommendation details.",
        "acceptance_use": "Proves whether semantic text similarity beats exact skill overlap.",
    },
    {
        "baseline": "simple_regression",
        "target": "job_fit_label_v2 and optional ats_label_v1 with separate feature sets",
        "allowed_inputs": "documented scalar features: skill overlap, cosine, experience gap, role match, requirement coverage, ATS component scores",
        "fit_rule": "Fit linear/ridge regression or logistic-style band model on training split only with fixed random seed.",
        "required_metrics": ["MAE", "RMSE", "R2", "Spearman", "score_band_agreement", "slice_metrics", "coefficient_review"],
        "leakage_control": "Features must be generated before manual validation labels are inspected; no identifiers or product copy.",
        "acceptance_use": "Best transparent comparator before neural scorer or feature-augmented scorer.",
    },
]

baseline_acceptance_policy = {
    "must_run_before_complex_training": True,
    "selection_gate": "A stronger model must improve MAE by at least 15-25% over the best valid baseline and show positive R2/Spearman before promotion beyond prototype.",
    "ranking_extension": "Recommendation experiments must also compare NDCG@5, NDCG@10, MAP@10, and candidate ordering examples against best ranking baseline.",
    "slice_requirement": "Report metrics by role family, language, experience band, pair type, and score band.",
}

required_baselines = {
    "constant_mean",
    "constant_median",
    "skill_overlap_only",
    "cosine_only",
    "simple_regression",
}
assert required_baselines == {row["baseline"] for row in baseline_model_catalog}

print_table(
    baseline_model_catalog,
    ["baseline", "target", "allowed_inputs", "fit_rule", "acceptance_use"],
)
print("\nBaseline acceptance policy:")
print(json.dumps(baseline_acceptance_policy, indent=2))


baseline           | target                                                                | allowed_inputs                                                                                                            | fit_rule                                                                                                | acceptance_use                                                               
-------------------+-----------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------
constant_mean      | job_fit_label_v2 and ats_label_v1 when labels exist                   | training labels only                                                                                     

## Step 2.5 — Manual validation plan

### Purpose
Describe how to sample low, medium, and high cases for human review without leaking validation labels into training.

### Required input
Phase 4 leakage-safe splits, score bands from this notebook, role/language/experience metadata from Phase 3, candidate pair metadata, and reviewer guidelines for job-fit and ATS labels.

### Action
Define sampling strata, minimum sample sizes, label fields, reviewer workflow, leakage controls, and report artifacts. Manual labels are validation evidence first; they must not be mixed into training labels without a new versioned label release.

### Expected output
A manual validation plan that covers low, medium, and high cases, documents reviewer process, and prevents validation labels from leaking into feature engineering or model fitting.

### Verification
Samples must be selected from locked validation/test groups only. Training code must not read manual-review columns unless a later phase explicitly promotes a new label version and rebuilds splits.


In [42]:
manual_validation_plan = {
    "label_manifest": {
        "artifact_name": "manual_validation_manifest_v1",
        "storage_policy": "Store review IDs, split name, band, role family, language, pair type, anonymized evidence pointers, reviewer labels, and adjudication status. Do not store raw contact PII.",
        "label_version": "jobfit_ats_manual_validation_v1",
        "primary_use": "validation_and_calibration_evidence_only",
    },
    "job_fit_sampling": {
        "minimum_bootstrap_cases_per_band": 30,
        "preferred_cases_per_band": 100,
        "bands": ["low", "medium", "high"],
        "strata": ["score_band", "role_family", "language", "experience_band", "pair_type"],
        "review_fields": [
            "skill_overlap_rating",
            "semantic_alignment_rating",
            "experience_match_rating",
            "role_match_rating",
            "requirement_coverage_rating",
            "final_job_fit_band",
            "reviewer_confidence",
            "notes",
        ],
        "sampling_rule": "Sample from locked validation/test groups after split assignment; include high-fit positives, medium-fit adjacent cases, hard negatives, and random negatives.",
    },
    "ats_sampling": {
        "minimum_bootstrap_cases_per_issue_family": 10,
        "preferred_cases_per_issue_family": 30,
        "case_families": [
            "normal_pdf",
            "scanned_pdf",
            "multi_column_pdf",
            "docx",
            "table_heavy_cv",
            "very_short_cv",
            "overly_long_cv",
        ],
        "review_fields": [
            "parseability_rating",
            "section_completeness_rating",
            "contact_detected",
            "dates_detected",
            "metric_evidence_rating",
            "formatting_risk_rating",
            "final_ats_band",
            "detected_issue_keys",
            "reviewer_confidence",
        ],
        "sampling_rule": "Use controlled benchmark files and real/synthetic CVs with documented license/consent; keep raw PII out of labels.",
    },
    "review_workflow": [
        "Freeze candidate IDs, split names, and source artifact hashes before review.",
        "Give reviewers only source evidence needed for the label, not model prediction internals or future validation labels.",
        "Use two independent reviewers for calibration subset and adjudicate disagreements.",
        "Record inter-reviewer agreement before changing label schema or score bands.",
        "Publish aggregate validation report without exposing raw CV contact data.",
    ],
    "leakage_controls": [
        "Manual labels cannot be used during Phase 3 normalization decisions for the same validation examples.",
        "Training code must ignore manual-review columns unless a later label version intentionally includes them.",
        "Group isolation by profile_id must remain intact after manual sampling.",
        "No wrapper-generated topActionables, sectionReviews, or hydrated job details may enter labels.",
        "Reviewer notes are qualitative audit data, not model features.",
    ],
}

phase2_acceptance = {
    "label_components_documented_before_pair_generation": bool(jobfit_label_components and ats_label_components),
    "baselines_defined_before_model_training": required_baselines == {row["baseline"] for row in baseline_model_catalog},
    "manual_validation_sample_plan_exists": all(
        key in manual_validation_plan for key in ["job_fit_sampling", "ats_sampling", "leakage_controls"]
    ),
}
assert all(phase2_acceptance.values()), phase2_acceptance

phase2_report = {
    "schema_version": "phase-02-label-schema-baselines-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_snapshot": "legacy",
    "inputs": {
        "phase1_report": str(PHASE1_REPORT.relative_to(ROOT)),
        "openapi": str(OPENAPI_JSON.relative_to(ROOT)),
        "todo_scope": "Phase 2 — Label Schema and Baseline Definitions",
    },
    "score_range": SCORE_RANGE,
    "openapi_score_constraints": openapi_score_constraints,
    "jobfit_label_components": jobfit_label_components,
    "jobfit_weight_sum": jobfit_weight_sum,
    "optional_jobfit_components": optional_jobfit_components,
    "ats_label_components": ats_label_components,
    "ats_weight_sum": ats_weight_sum,
    "score_band_policy": score_band_policy,
    "score_policy_notes": score_policy_notes,
    "baseline_model_catalog": baseline_model_catalog,
    "baseline_acceptance_policy": baseline_acceptance_policy,
    "manual_validation_plan": manual_validation_plan,
    "blocked_until_later_phases": [
        "Phase 3 must define normalization rules before labels are generated.",
        "Phase 4 must generate balanced pairs with leakage-safe splits before baseline evaluation.",
        "Phase 5 must run this baseline catalog before model training.",
        "Phase 7 must build controlled ATS benchmark samples before production ATS claims.",
        "Phase 10 must calibrate score bands before production score semantics are finalized.",
    ],
    "acceptance": phase2_acceptance,
}

write_json(PHASE2_REPORT, phase2_report)
print(f"Wrote {PHASE2_REPORT.relative_to(ROOT)}")
print(json.dumps(phase2_acceptance, indent=2))


Wrote reports/phase_02_label_schema_baselines.json
{
  "label_components_documented_before_pair_generation": true,
  "baselines_defined_before_model_training": true,
  "manual_validation_sample_plan_exists": true
}


## Acceptance criteria

- [x] Label components are documented before pair generation.
- [x] Baselines are defined before model training.
- [x] Manual validation sample plan exists.


## Phase notes

Phase 2 defines schemas, policies, and validation gates only. It intentionally does not generate new pairs, train a model, calibrate thresholds, or claim production readiness.

Follow-up work:

- Phase 3 must implement normalization rules for experience, skills, language, and text construction before label generation.
- Phase 4 must create balanced pair metadata and leakage-safe splits using this label schema.
- Phase 5 must execute the baseline catalog and compare all future models against the best valid baseline.
- ATS labels remain blocked for production claims until controlled CV benchmark coverage exists.
